# 4 · Memoria y «aprendizaje»

**Objetivo:** que el agente **guarde conocimiento nuevo** durante el uso y lo **reutilice** en
conversaciones futuras. Su comportamiento cambia sin reentrenar ningún modelo.

**Empezar aquí sin haber hecho las secciones anteriores:** la recuperación híbrida de la sección 3 se
importa ya hecha con `build_retriever()`.

In [ ]:
from taller import get_llm
from taller.agents import build_retriever
from taller.viz import mostrar_grafo, mostrar_mensajes

llm = get_llm()
retriever = build_retriever()  # BM25 + embeddings sobre la documentación de Cuac & Co.

## 4.1 · Dos tipos de memoria

| | **Corto plazo** | **Largo plazo** |
|---|---|---|
| Qué guarda | Los mensajes de **una** conversación | Hechos, preferencias... **entre** conversaciones |
| Pieza de LangGraph | *Checkpointer* (`InMemorySaver`, `SqliteSaver`, `PostgresSaver`) | *Store* (`InMemoryStore`, `SqliteStore`, `PostgresStore`) |
| Se identifica por | `thread_id` | `namespace` (una tupla, p. ej. `("usuarios", "ana")`) |

### Corto plazo: hilos (*threads*)

Con un *checkpointer*, LangGraph guarda el estado tras cada paso. Cada `thread_id` es una conversación.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

from taller.agents import build_basic_agent

agente = build_basic_agent(checkpointer=InMemorySaver())

hilo_1 = {"configurable": {"thread_id": "hilo-1"}}
agente.invoke({"messages": [("human", "Hola, me llamo Ana y trabajo en el equipo de Pagos.")]}, hilo_1)
print(agente.invoke({"messages": [("human", "¿Cómo me llamo?")]}, hilo_1)["messages"][-1].text)

hilo_2 = {"configurable": {"thread_id": "hilo-2"}}
print(agente.invoke({"messages": [("human", "¿Cómo me llamo?")]}, hilo_2)["messages"][-1].text)

En el hilo 2 no lo sabe: la memoria de corto plazo **no se comparte entre conversaciones**.
Para eso está el *store*.

## 4.2 · Largo plazo: el *store*

Un almacén clave-valor organizado por *namespaces*. Si le damos un modelo de embeddings, además
permite **búsqueda semántica**. `build_memory_store()` lo configura según tu proveedor.

In [ ]:
from taller.agents import build_memory_store

store = build_memory_store()  # InMemoryStore(index={"embed": ..., "dims": ..., "fields": ["texto"]})

store.put(("demo",), "hecho-1", {"texto": "El equipo de Pagos come a las 14:00"})
store.put(("demo",), "hecho-2", {"texto": "La demo general es el último jueves de mes"})

for item in store.search(("demo",), query="¿a qué hora almuerza Pagos?", limit=2):
    print(f"{item.score}  {item.value['texto']}")

## 4.3 · Herramientas con memoria

Las herramientas pueden recibir un parámetro `runtime: ToolRuntime`. **LangGraph lo inyecta y el LLM
no lo ve.** Da acceso a:

- `runtime.store`: el *store* de largo plazo.
- `runtime.context`: el **contexto** de la ejecución (aquí, quién es el usuario), que se pasa en
  `invoke(..., context=...)`.

### ✍️ Ejercicio 5: completa `aprender_hecho`

In [ ]:
import uuid
from datetime import date

from langchain.tools import ToolRuntime, tool

from taller.tools import Context  # @dataclass con un campo: user_id

KNOWLEDGE_NS = ("conocimiento",)


@tool
def aprender_hecho(hecho: str, runtime: ToolRuntime[Context]) -> str:
    """Guarda un hecho nuevo o actualizado sobre la empresa para futuras conversaciones.

    Úsala cuando el usuario te informe de algo que no está en la documentación o que la contradice.
    Escribe el hecho como una frase autocontenida, con fechas si las hay.
    """
    value = {"texto": hecho, "autor": runtime.context.user_id, "fecha": date.today().isoformat()}
    # TODO: guarda `value` en runtime.store, en el namespace KNOWLEDGE_NS, con una clave única (uuid)
    ...
    return f"Guardado: {hecho}"

<details><summary><b>Ver solución</b></summary>

```python
    runtime.store.put(KNOWLEDGE_NS, uuid.uuid4().hex, value)
```
</details>

El resto de herramientas ya está hecho en `taller/tools.py`:

- `buscar_documentacion`: busca en la documentación **y** en lo aprendido, y fusiona ambos con RRF.
- `guardar_preferencia`: guarda cómo quiere el usuario que le respondan, en `("usuarios", user_id)`.

In [ ]:
from taller.tools import hora_actual, make_memory_tools

herramientas = make_memory_tools(retriever)  # diccionario nombre → herramienta
herramientas["aprender_hecho"] = aprender_hecho  # la tuya; comenta esta línea para usar la de referencia
tools = list(herramientas.values())

## 4.4 · El agente que aprende

Dos novedades respecto al agente de la sección 2:

1. `StateGraph(MessagesState, context_schema=Context)`: el grafo declara qué contexto recibe.
2. El nodo `agent` recibe `runtime: Runtime[Context]` y **lee las preferencias del usuario del store**
   para incluirlas en el prompt de sistema. La memoria entra en el prompt en cada turno.

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime

from taller.agents import SYSTEM_PROMPT

llm_con_tools = llm.bind_tools(tools)


def agent(state: MessagesState, runtime: Runtime[Context]) -> dict:
    prefs = runtime.store.get(("usuarios", runtime.context.user_id), "preferencias")
    system = SYSTEM_PROMPT.format(
        today=hora_actual.invoke({}),
        preferences="\n".join(f"- {p}" for p in prefs.value["lista"]) if prefs else "(ninguna)",
    )
    return {"messages": [llm_con_tools.invoke([SystemMessage(system), *state["messages"]])]}


builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

store = build_memory_store()
agente = builder.compile(checkpointer=InMemorySaver(), store=store)
mostrar_grafo(agente)
print(SYSTEM_PROMPT)

Una función para conversar: cada llamada indica el hilo y el usuario.

In [ ]:
def chat(mensaje: str, thread_id: str, user_id: str) -> None:
    result = agente.invoke(
        {"messages": [("human", mensaje)]},
        {"configurable": {"thread_id": thread_id}},
        context=Context(user_id=user_id),
    )
    mostrar_mensajes(result["messages"][-1:])

## 4.5 · Ver cómo aprende

**Conversación 1.** Ana pregunta por la congelación de despliegues. El agente responde con la documentación.

In [ ]:
chat("¿Puedo desplegar a producción un jueves a las 19:00?", thread_id="c1", user_id="ana")

Ana informa de un cambio que **todavía no está en la documentación**:

In [ ]:
chat(
    "Te aviso: desde el 1 de noviembre de 2026 las ventanas de despliegue de lunes a jueves "
    "se amplían hasta las 20:00.",
    thread_id="c1",
    user_id="ana",
)

**Conversación 2**, nueva y con **otro usuario**. Luis no ha visto nada de lo anterior.

In [ ]:
chat("¿Puedo desplegar a producción un jueves a las 19:00?", thread_id="c2", user_id="luis")

El agente ha **cambiado de comportamiento** porque `buscar_documentacion` ahora también devuelve lo
aprendido, marcado como `[aprendido]` y con fecha. Veamos qué hay en el store:

In [ ]:
for item in store.search(KNOWLEDGE_NS):
    print(item.value)

### Preferencias por usuario

Las preferencias se guardan en el *namespace* de cada usuario y se inyectan en el prompt de sistema.

In [ ]:
chat("A partir de ahora respóndeme siempre en una sola frase, sin listas.", thread_id="c3", user_id="luis")
chat("¿Cómo funcionan las guardias?", thread_id="c4", user_id="luis")  # hilo nuevo, misma persona

## 4.6 · En Studio: el mismo agente, con interfaz

El servidor sirve este agente como **agente_que_aprende** (con la aprobación humana activada: la veremos en la
sección 5). Dos cosas que en Studio se ven mejor que en el notebook:

- **Contexto:** en el panel de configuración puedes poner `user_id` (Ana, Luis...) antes de cada ejecución.
- **Memoria:** el *store* del servidor se consulta desde la pestaña de memoria a largo plazo, por namespace:
  `conocimiento` (lo aprendido) y `usuarios/<user_id>` (preferencias).

Desde Python, el servidor se usa igual: el `context` viaja en la petición. Ojo: el servidor tiene **su propio
store**, distinto del de este notebook, así que de entrada responderá con lo que dice la documentación; enséñale
el cambio de ventana en Studio (con aprobación) y vuelve a preguntar.

In [ ]:
from taller.servidor import preguntar

preguntar("agente_que_aprende", "¿Cuál es la ventana de despliegue de los jueves?", user_id="luis")

## 4.7 · Lo que hay que vigilar

Dejar que un agente escriba en su propia memoria es potente, y también arriesgado:

- **Envenenamiento de memoria:** cualquiera puede «enseñarle» algo falso. → En la sección 5 añadimos
  **aprobación humana** antes de guardar.
- **Contradicciones:** ¿qué gana, la documentación o lo aprendido? → Hecho explícito en el prompt:
  gana lo más reciente, y se dice.
- **Privacidad:** separar *namespaces* por usuario (`("usuarios", user_id)`) y por organización.
- **Olvido:** los stores admiten TTL (caducidad) para que la memoria no crezca para siempre.

> **Para recordar:** «aprender» aquí significa **escribir en una memoria que el agente consulta**.
> El modelo no cambia: cambia el contexto que recibe. Es barato, inmediato, auditable y reversible.

### 🚀 Reto opcional: memoria «en segundo plano»

Ahora el LLM decide en caliente cuándo guardar. Otra opción es un nodo `reflexionar` que se ejecute
**al final** de cada conversación, extraiga hechos con salida estructurada (`llm.with_structured_output`)
y los guarde. Pista: cambia la arista condicional para que, en vez de ir a `END`, vaya a `reflexionar`.